# Colab — Hard negative mining (pipeline clásico)

Re-corre el detector sobre las imágenes train, identifica falsos positivos (proposals que la SVM marcó como producto pero realmente eran fondo), los añade al training set como negativos extra y re-entrena la SVM. Repite por `hard_negative.rounds` rondas (default 2).

**Tarda** comparable a `colab_build_features.ipynb` × nº rondas, porque cada ronda re-extrae features sobre todas las imgs train.

**Resume**:
- A nivel ronda: `.hardneg_state.json` guarda `completed_rounds`. Si Colab cae después de la ronda 1, al relanzar empieza por la ronda 2.
- Dentro de una ronda: checkpoint parcial cada 100 imgs (`classical_features.hardneg_partial.npz`).

## Prerequisitos

En Drive `/MyDrive/grocery-detection/processed/` debe haber ya:
- `train.json`
- `codebook.pkl`
- `classical_features.npz`  (de colab_build_features.ipynb)
- `classical_svm.pkl`        (de colab_train_svm.ipynb + import_colab_svm.py — o equivalente)

## 1. Bootstrap

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    sync_from_drive, sync_to_drive, run_script,
)
print("helpers cargados")

In [ ]:
mount_drive()
install_deps()

In [ ]:
setup_dataset()

## 2. Bajar artifacts desde Drive

Incluye state y checkpoint parcial — si existían, reanuda.

In [ ]:
sync_from_drive([
    "train.json",
    "codebook.pkl",
    "classical_features.npz",
    "classical_svm.pkl",
    ".hardneg_state.json",
    "classical_features.hardneg_partial.npz",
])

## 3. Mining + refit

Si quieres forzar empezar desde la ronda 1, descomenta el `--reset`.

In [ ]:
run_script("scripts/run_classical_hard_neg.py")
# run_script("scripts/run_classical_hard_neg.py", "--reset")

## 4. Subir artifacts actualizados a Drive

In [ ]:
sync_to_drive([
    "classical_features.npz",
    "classical_svm.pkl",
    ".hardneg_state.json",
])

## Siguiente paso

Cuando todas las rondas estén completas (mira el log final), abre `notebooks/colab_infer.ipynb` para predecir sobre el test split.